# Horse Health

In this notebook, predictive analytics study is performed on a Kaggle Playground Series 3.22 data set which consists of health conditions of horses.

The breakdown of the study is as below:

<ul>
<li>Data exploration</li>
<li>Data transformation</li>
<li>Feature selection</li>
<li>Model construction</li>
</ul>

Competition Score: 0.75

In [ ]:
import warnings
import numpy as numpy
import pandas as pandas
import matplotlib.pyplot as pyplot
from sklearn.preprocessing import OrdinalEncoder
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree

In [ ]:
warnings.filterwarnings('ignore')
pandas.set_option('display.max_columns', None)
pandas.set_option('display.max_rows', None)

# Data Exploration

<ul>
<li>Extract a sample from data set</li>
<li>Get information about data set</li>
<li>Count the number of unique values for each column</li>
<li>Classify variables with the help of viewed sample, data type information and number of unique values</li>
<li>Investigate the distribution of target variable</li>
</ul>

In [ ]:
TrainData = pandas.read_csv('/kaggle/input/playground-series-s3e22/train.csv')
TestData = pandas.read_csv('/kaggle/input/playground-series-s3e22/test.csv')
TrainData.sample(10)

In [ ]:
TrainData.info()

In [ ]:
TrainData.nunique()

In [ ]:
TargetVariable = 'outcome'

CategoricalVariables = ['surgery', 'age', 'temp_of_extremities', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time', 'pain', 'peristalsis', 
                        'abdominal_distention', 'nasogastric_tube', 'nasogastric_reflux', 'rectal_exam_feces', 'abdomen', 'abdomo_appearance', 'surgical_lesion', 'cp_data']

ContinuousVariables = ['rectal_temp', 'pulse', 'respiratory_rate', 'nasogastric_reflux_ph', 'packed_cell_volume', 'total_protein', 'abdomo_protein']

In [ ]:
TrainData[TargetVariable].value_counts().plot(kind='pie', title='Target Variable: Health Outcomes')

# Data Transformation

<ul>
<li>Fill missing values with the mode value of each categorical column in both training and test data set</li>
<li>Fill missing values with the mean value of each continuous column in both training and test data set</li>
<li>Map the nominal values of target variable with integer values to construct model properly</li>
<li>Use ordinal encoding to process the values of categorical values to make feature selection properly</li>
<li>Extract a sample from transformed data set</li>
<li>Get information about transformed data set</li>
</ul>

In [ ]:
for variable in CategoricalVariables:
    TrainData[variable].fillna(TrainData[variable].mode()[0], inplace=True)

for variable in CategoricalVariables:
    TestData[variable].fillna(TestData[variable].mode()[0], inplace=True)

for variable in ContinuousVariables:
    TrainData[variable].fillna(TrainData[variable].mean(), inplace=True)

for variable in ContinuousVariables:
    TestData[variable].fillna(TestData[variable].mean(), inplace=True)

TrainData[TargetVariable] = TrainData[TargetVariable].map({'lived':1, 'euthanized':2, 'died': 3})

ordinalEncoder = OrdinalEncoder()
TrainData[CategoricalVariables] = ordinalEncoder.fit_transform(TrainData[CategoricalVariables])
TestData[CategoricalVariables] = ordinalEncoder.fit_transform(TestData[CategoricalVariables])

TrainData.sample(10)

In [ ]:
TrainData.info()

# Feature Selection

Target variable of this study is categorical and there is both categorical and continuous variables among candidate features. During feature selection step, we have to seek for a variation of the pattern between target variable and candidate features. Visualization methods can help us to make investigation however the relationship between variables should be statistically proven.

<ul>
<li> Investigate the relationship between target variable and categorical variables
    <ul>
    <li>Make cross tab to count target variable for each categorical variable</li>
    <li>Plot the cross tab values as a bar chart for visual evaluation</li>
    <li>Perform Chi Square Test for statistical evaluation</li>
    </ul>
</li>
<li> Investigate the relationship between target variable and continuous variables
    <ul>
    <li>Plot a Boxplot of each variable for visual evaluation</li>
    <li>Perform ANOVA Test for statistical evaluation</li>
    </ul>
</li>
</ul>

In [ ]:
for variable in CategoricalVariables:
    CrossTabDataFrame = pandas.crosstab(index=TrainData[variable], columns=TrainData[TargetVariable])
    CrossTabDataFrame.plot.bar(title=variable)
    ChiSquareTest = chi2_contingency(CrossTabDataFrame)
    if (ChiSquareTest[1] < 0.05):
        print(variable, "is correlated with target variable\nP Value:", ChiSquareTest[1])
    else:
        print(variable, "is -NOT- correlated with target variable\nP Value:", ChiSquareTest[1])

In [ ]:
for variable in ContinuousVariables:
    TrainData.boxplot(column=variable, by=TargetVariable)
    AnovaTest = f_oneway(TrainData[TargetVariable], TrainData[variable])
    if (AnovaTest[1] < 0.05):
        print(variable, "is correlated with target variable\nP Value:", AnovaTest[1])
    else:
        print(variable, "is -NOT- correlated with target variable\nP Value:", AnovaTest[1])

# Random Forest

Random Forest Classification is an appropriate algorithm to solve this problem.

<ul>
<li>Set predictor variables after feature selection</li>
<li>Split data set into train and test</li>
<li>Find best scoring fold number using cross validation</li>
<li>Find best hyperparameters using grid search</li>
<li>Build random forest with best hyperparameters</li>
<li>Make predictions</li>
<li>Evaluate model performance</li>
<li>Visualize some of the trees in random forest</li>
</ul>

In [ ]:
PredictorVariables = []

for variable in CategoricalVariables:
    PredictorVariables.append(variable)

for variable in ContinuousVariables:
    PredictorVariables.append(variable)

X = TrainData[PredictorVariables]
Y = TrainData[TargetVariable]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state=42)

RandomForest = RandomForestClassifier()

FoldScores = []

for i in range(2,11):
    FoldScores.append(cross_val_score(RandomForest, X_train, Y_train, cv=i, scoring='f1_micro').sum() / i)

parameters = {'criterion': ['entropy', 'gini'],
              'max_depth': [4, 6, 8, 10],
              'n_estimators': [100, 125, 150, 175, 200],
              'random_state': [42]}

gridSearch = GridSearchCV(estimator=RandomForest, param_grid=parameters, cv=FoldScores.index(max(FoldScores)) + 2, scoring='f1_micro')

gridSearch.fit(X_train, Y_train)

print("Highest scoring fold number:", FoldScores.index(max(FoldScores)) + 2, "(Accuracy: %", max(FoldScores), ")")
print("Best Hyperparameters", gridSearch.best_params_)

In [ ]:
RandomForest = RandomForestClassifier(**gridSearch.best_params_)

RandomForest.fit(X_train, Y_train)

Y_pred = RandomForest.predict(X_test)

print(metrics.classification_report(Y_test, Y_pred))

In [ ]:
fig, axs = pyplot.subplots(nrows = 1, ncols = 3, figsize = (10,2), dpi=900)

for i in range(0, 3):
    plot_tree(RandomForest.estimators_[i], filled = True, ax = axs[i])
    axs[i].set_title('Tree ' + str(i + 1), fontsize = 8)

In [ ]:
X_submission = TestData[PredictorVariables]

Y_submission = RandomForest.predict(X_submission)

Y_submission = pandas.DataFrame(Y_submission)
Y_submission[0] = Y_submission[0].map({1: 'lived', 2: 'euthanized', 3:'died'})

Submission = pandas.DataFrame({'id': TestData.id, 'outcome': Y_submission[0]})
Submission.to_csv('Submission.csv', index=False)